In [ ]:
import numpy as np 
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

In [ ]:
!git clone https://github.com/Algolzw/daclip-uir.git /kaggle/working/daclip-uir
%cd /kaggle/working/daclip-uir
!ls

In [ ]:
!pip install -q open_clip_torch ftfy regex lmdb pyyaml tqdm opencv-python scikit-image lpips huggingface_hub
!cat requirements.txt   # check for anything missing, but skip torch/torchvision pins

In [ ]:
import os
from huggingface_hub import hf_hub_download

os.makedirs("universal-image-restoration/pretrained", exist_ok=True)
for f in ["daclip_ViT-B-32.pt", "universal-ir.pth"]:
    hf_hub_download("weblzw/daclip-uir-ViT-B-32-irsde", f,
                    local_dir="universal-image-restoration/pretrained")
!ls universal-image-restoration/pretrained

In [ ]:
!ls /kaggle/input/datasets/

In [ ]:
!ls /kaggle/input
!ls /kaggle/input/datasets/joe1995/div2k-dataset/

In [ ]:
import os

base_path = '/kaggle/input/datasets/'
print("Searching for image folders...")

for root, dirs, files in os.walk(base_path):
    png_count = sum(1 for f in files if f.endswith('.png'))
    if png_count > 0:
        print(f"Found {png_count} images in: {root}")

In [ ]:
import os
import cv2
import glob

# Using the exact nested path for the training set
input_dir = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'
output_dir = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
scale_factor = 4  # Change to 2 or 3 if needed

os.makedirs(output_dir, exist_ok=True)

# Fetch all HR image paths
hr_images = glob.glob(os.path.join(input_dir, '*.png'))
print(f"Found {len(hr_images)} images. Starting compression...")

for img_path in hr_images:
    filename = os.path.basename(img_path)
    img = cv2.imread(img_path)
    
    if img is None:
        continue
        
    # Calculate new dimensions
    height, width = img.shape[:2]
    new_width = width // scale_factor
    new_height = height // scale_factor
    
    # Downsample using bicubic interpolation
    lr_img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_CUBIC)
    
    # Save the compressed image
    output_path = os.path.join(output_dir, filename)
    cv2.imwrite(output_path, lr_img)

print(f"Compression complete. LR images saved to {output_dir}")

In [ ]:
import yaml
import os

config_path = "universal-image-restoration/config/daclip-sde/options/test.yml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Point the dataloader to your compressed DIV2K dataset
config['datasets']['test']['dataroot_lq'] = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
# If you want to calculate metrics (PSNR/SSIM), point to the original HR Kaggle input path
config['datasets']['test']['dataroot_gt'] = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'

# Ensure it points to your downloaded pretrained weights
config['path']['pretrain_model_G'] = 'universal-image-restoration/pretrained/universal-ir.pth'
config['path']['daclip'] = 'universal-image-restoration/pretrained/daclip_ViT-B-32.pt'

with open(config_path, 'w') as f:
    yaml.dump(config, f)
    
print("Updated test.yml with Kaggle paths!")

In [ ]:
import yaml
import os

config_path = "universal-image-restoration/config/daclip-sde/options/test.yml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Get the actual test dataset key (usually 'test_1' or 'val')
test_keys = list(config['datasets'].keys())

for key in test_keys:
    # Update all test dataset paths found in the config
    config['datasets'][key]['dataroot_lq'] = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
    config['datasets'][key]['dataroot_gt'] = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'

# Ensure it points to your downloaded pretrained weights
config['path']['pretrain_model_G'] = 'universal-image-restoration/pretrained/universal-ir.pth'
config['path']['daclip'] = 'universal-image-restoration/pretrained/daclip_ViT-B-32.pt'

with open(config_path, 'w') as f:
    yaml.dump(config, f)
    
print(f"Updated test.yml successfully! Replaced paths for: {test_keys}")

In [ ]:
%cd universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

In [ ]:
import yaml

config_path = "/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde/options/test.yml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

test_keys = list(config['datasets'].keys())

for key in test_keys:
    # Update the correct uppercase keys
    config['datasets'][key]['dataroot_LQ'] = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
    config['datasets'][key]['dataroot_GT'] = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'
    
    # Remove the incorrect lowercase keys
    config['datasets'][key].pop('dataroot_lq', None)
    config['datasets'][key].pop('dataroot_gt', None)

# Set absolute paths for pretrained models
config['path']['pretrain_model_G'] = '/kaggle/working/daclip-uir/universal-image-restoration/pretrained/universal-ir.pth'
config['path']['daclip'] = '/kaggle/working/daclip-uir/universal-image-restoration/pretrained/daclip_ViT-B-32.pt'

with open(config_path, 'w') as f:
    yaml.dump(config, f)
    
print("Fixed YAML config with uppercase keys!")

In [ ]:
!pip install ema_pytorch

In [ ]:
!sed -i 's/checkpoint = torch.load(checkpoint_path, map_location=map_location)/checkpoint = torch.load(checkpoint_path, map_location=map_location, weights_only=False)/g' /kaggle/working/daclip-uir/universal-image-restoration/open_clip/factory.py

In [ ]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

In [ ]:
import os
import random
import glob

lr_dir = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
all_images = sorted(glob.glob(os.path.join(lr_dir, '*.png')))

# Seed for reproducibility so you get the same 300 images if you rerun this
random.seed(42)

if len(all_images) > 300:
    # Randomly select 300 images to keep
    images_to_keep = set(random.sample(all_images, 300))
    
    deleted_count = 0
    for img_path in all_images:
        if img_path not in images_to_keep:
            os.remove(img_path)
            deleted_count += 1
            
    print(f"Deleted {deleted_count} images. Exactly 300 diverse images remain in the folder.")
else:
    print(f"The folder currently has {len(all_images)} images. No deletion needed.")

In [ ]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

In [ ]:
import os
import shutil
import glob
import yaml

lr_dir = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
gt_original_dir = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'
gt_subset_dir = '/kaggle/working/DIV2K_train_HR_subset'

os.makedirs(gt_subset_dir, exist_ok=True)

# Fetch the 300 remaining LR image filenames
lr_images = glob.glob(os.path.join(lr_dir, '*.png'))

# Copy the corresponding HR images to a new working folder
for lr_path in lr_images:
    filename = os.path.basename(lr_path)
    src_gt = os.path.join(gt_original_dir, filename)
    dst_gt = os.path.join(gt_subset_dir, filename)
    if os.path.exists(src_gt):
        shutil.copy(src_gt, dst_gt)

print(f"Copied {len(os.listdir(gt_subset_dir))} matching GT images to {gt_subset_dir}")

# Update test.yml to point to the new 300-image GT folder
config_path = "/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde/options/test.yml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

for key in list(config['datasets'].keys()):
    config['datasets'][key]['dataroot_GT'] = gt_subset_dir

with open(config_path, 'w') as f:
    yaml.dump(config, f)
    
print("Updated config to use the new matching subset!")

In [ ]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

In [ ]:
import os

test_py_path = '/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde/test.py'

with open(test_py_path, 'r') as f:
    code = f.read()

# The exact replacement code that forces resizing before scoring
patch_code = """
            if cropped_sr_img.shape != cropped_gt_img.shape:
                import cv2
                cropped_gt_img = cv2.resize(cropped_gt_img, (cropped_sr_img.shape[1], cropped_sr_img.shape[0]))
            psnr = util.calculate_psnr(cropped_sr_img * 255, cropped_gt_img * 255)
"""

# Replace the specific PSNR calculation line
code = code.replace(
    "psnr = util.calculate_psnr(cropped_sr_img * 255, cropped_gt_img * 255)", 
    patch_code.strip()
)

with open(test_py_path, 'w') as f:
    f.write(code)

print("test.py has been successfully patched to resize ground truth images!")

In [ ]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

In [ ]:
test_py_path = '/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde/test.py'

with open(test_py_path, 'r') as f:
    code = f.read()

# Resize the PyTorch GT tensor to match SR_img's spatial dimensions (H, W) before LPIPS
old_lpips = "lp_score = lpips_fn("
new_lpips = """if GT.shape[-2:] != SR_img.shape[-2:]:
                GT = torch.nn.functional.interpolate(GT, size=SR_img.shape[-2:], mode='bilinear', align_corners=False)
            lp_score = lpips_fn("""

if "torch.nn.functional.interpolate(GT" not in code:
    code = code.replace(old_lpips, new_lpips)
    with open(test_py_path, 'w') as f:
        f.write(code)
    print("Patched LPIPS tensor shape mismatch in test.py!")
else:
    print("Patch already applied.")

In [ ]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

In [ ]:
import os
import cv2
import glob

lq_dir = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
gt_dir = '/kaggle/working/DIV2K_train_HR_subset'

lq_files = sorted(glob.glob(os.path.join(lq_dir, '*.png')))

for lq_path in lq_files:
    fname = os.path.basename(lq_path)
    gt_path = os.path.join(gt_dir, fname)
    
    if os.path.exists(gt_path):
        lq_img = cv2.imread(lq_path)
        gt_img = cv2.imread(gt_path)
        
        # Step 1 (Option A): Resize GT to match LQ's exact 1:1 dimensions
        h, w = lq_img.shape[:2]
        gt_resized = cv2.resize(gt_img, (w, h), interpolation=cv2.INTER_AREA)
        cv2.imwrite(gt_path, gt_resized)
        
        # Step 2 (Option B): Apply Paper's JPEG Compression (Quality=10) to create true LQ
        encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 10]
        _, encimg = cv2.imencode('.jpg', gt_resized, encode_param)
        jpeg_degraded_img = cv2.imdecode(encimg, 1)
        cv2.imwrite(lq_path, jpeg_degraded_img)

print(f"Prepared {len(lq_files)} aligned 1:1 GT and JPEG-degraded (QF=10) LQ pairs!")

In [ ]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml